# 📓 TF-IDF Keyword Extraction & Search Mode Ablation

**Autor:** Sakina Ahmadi
**Beschreibung:** Dieses Notebook dokumentiert die TF-IDF Keyword-Extraktion
und den Vergleich dreier Retrieval-Modi: Pure Dense, Pure Keyword und Hybrid.

## Ziel
1. **TF-IDF Keyword-Extraktion** – Pro Chunk die Top-10 Keywords extrahieren
2. **Qdrant Full-Text Index** – In-Engine Word Tokenizer initialisieren
3. **Drei Suchmodi vergleichen** – Pure Dense vs. Pure Keyword vs. Hybrid
4. **Ergebnisse analysieren** – Hit Rate, MRR, Latenz

---
## 1. Server-Konfiguration & Architektur

Die Analyse der produktiven Serverkonfiguration des Qdrant Cloud-Clusters
(`stage2_docling_hybrid_bge`) offenbarte drei zentrale Erkenntnisse:

1. **Unbenannter Dense-Vektorraum:** 1024-dimensionale Vektoren (BGE-M3)
2. **Keine spärliche Vektormatrix:** BM25 ist nicht aktiv
3. **Kundenspezifischer Volltext-Index:** `text_llm` mit Word Tokenizer

In [ ]:
# =====================================================================
# 1. SERVER-KONFIGURATION ABRUFEN
# =====================================================================
from qdrant_client import QdrantClient
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
url = user_secrets.get_secret("QDRANT_URL")
api_key = user_secrets.get_secret("QDRANT_API_KEY")

client = QdrantClient(url=url, api_key=api_key, check_compatibility=False)

COLLECTION_NAME = "stage2_docling_hybrid_bge"
collection_info = client.get_collection(collection_name=COLLECTION_NAME)

print(f"Status: {collection_info.status}")
print(f"Points: {collection_info.points_count:,}")
print(f"Vector size: {collection_info.config.params.vectors.size}")
print(f"Distance: {collection_info.config.params.vectors.distance}")

---
## 2. TF-IDF Keyword-Extraktion

Wir extrahieren pro Chunk die Top-10 Keywords mittels TF-IDF.
Diese Keywords werden später für den lexikalischen Filter verwendet.

In [ ]:
# =====================================================================
# 2. TF-IDF KEYWORD-EXTRAKTION
# =====================================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords

STOPWORDS_EN = set(stopwords.words("english"))

def extract_chunks_from_markdown(md_text):
    """Extrahiert alle CHUNK CONTENT Blöcke aus einem Markdown-Paper."""
    chunks = re.findall(r"CHUNK CONTENT:(.*?)(?=CHUNK CONTENT:|$)", md_text, flags=re.DOTALL)
    return [chunk.strip() for chunk in chunks]

def extract_tfidf_keywords_per_chunk(chunk_text, top_k=10):
    """Extrahiert TF-IDF Keywords aus einem einzelnen Chunk."""
    vectorizer = TfidfVectorizer(
        stop_words="english",
        token_pattern=r"[A-Za-z][A-Za-z\-]{2,}",
        max_features=5000
    )
    tfidf_matrix = vectorizer.fit_transform([chunk_text])
    scores = tfidf_matrix.toarray()[0]
    words = vectorizer.get_feature_names_out()
    scored_words = list(zip(words, scores))
    scored_words.sort(key=lambda x: x[1], reverse=True)
    return [w for w, s in scored_words[:top_k]]

print("✅ TF-IDF Keyword-Extraktion definiert")

---
## 3. Qdrant Full-Text Index initialisieren

Wir erstellen einen In-Engine Word Tokenizer auf dem `text_llm`-Feld.
Dies ermöglicht lexikalische Suchen ohne separate Sparse-Matrix.

In [ ]:
# =====================================================================
# 3. QDRANT VOLLTEXT-INDEX INITIALISIEREN
# =====================================================================
from qdrant_client import models

try:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="text_llm",
        field_schema=models.TextIndexParams(
            type=models.TextIndexType.TEXT,
            tokenizer=models.TokenizerType.WORD,
            lowercase=True,
            min_token_len=2,
            max_token_len=20
        )
    )
    print("✅ Full-Text Index erfolgreich erstellt")
except Exception as e:
    print(f"⚠️ Index existiert bereits oder Fehler: {e}")

---
## 4. Drei Suchmodi definieren

Wir vergleichen drei Retrieval-Strategien:
1. **Pure Dense** – Nur semantische Vektor-Ähnlichkeitssuche
2. **Pure Keyword** – Nur lexikalische Suche (TF-IDF Keywords)
3. **Dense + Keyword Hybrid** – Kombination aus beidem

In [ ]:
# =====================================================================
# 4. DREI SUCHMODI DEFINIEREN
# =====================================================================
import requests
import time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3", device="cuda")

base_url = url.rstrip('/')
search_endpoint = f"{base_url}/collections/{COLLECTION_NAME}/points/search"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

SUCH_MODI = ["Modus_1_Pure_Dense", "Modus_2_Pure_Keyword", "Modus_3_Dense_Keyword_Hybrid"]

def search_pure_dense(query_text, k=10):
    """Modus 1: Nur semantische Suche."""
    raw_vector = model.encode([query_text])[0]
    dense_vector = raw_vector.tolist() if hasattr(raw_vector, "tolist") else list(raw_vector)
    
    json_payload = {
        "vector": dense_vector,
        "limit": k,
        "with_payload": True
    }
    
    response = requests.post(search_endpoint, json=json_payload, headers=headers, timeout=30)
    return response.json().get("result", [])

def search_pure_keyword(query_text, k=10):
    """Modus 2: Nur lexikalische Suche."""
    json_payload = {
        "vector": [0.0] * 1024,
        "filter": {
            "must": [{"key": "text_llm", "match": {"text": query_text}}]
        },
        "limit": k,
        "with_payload": True
    }
    
    response = requests.post(search_endpoint, json=json_payload, headers=headers, timeout=30)
    return response.json().get("result", [])

def search_hybrid(query_text, k=10):
    """Modus 3: Dense + Keyword Hybrid."""
    raw_vector = model.encode([query_text])[0]
    dense_vector = raw_vector.tolist() if hasattr(raw_vector, "tolist") else list(raw_vector)
    
    json_payload = {
        "vector": dense_vector,
        "filter": {
            "must": [{"key": "text_llm", "match": {"text": query_text}}]
        },
        "limit": k,
        "with_payload": True
    }
    
    response = requests.post(search_endpoint, json=json_payload, headers=headers, timeout=30)
    return response.json().get("result", [])

print("✅ Drei Suchmodi definiert")

---
## 5. Evaluierung der drei Suchmodi

Wir evaluieren alle drei Modi mit 99 Gold-Standard-Fragen
und berechnen Hit Rate, MRR und Latenz.

In [ ]:
# =====================================================================
# 5. EVALUIERUNG DER DREI SUCHMODI
# =====================================================================
import json
import mlflow

# Gold-Standard laden
with open("/kaggle/input/datasets/sakinaahmadi/automl-ground-truth-100/automl_ground_truth_100.json", "r") as f:
    gold_standard = json.load(f)

queries = [item["query"] for item in gold_standard]
target_ids = [item["arxiv_id"].strip() for item in gold_standard]

mlflow.set_experiment("Search_Mode_Ablation")

for modus in SUCH_MODI:
    print(f"\n🔬 Teste: {modus}")
    
    with mlflow.start_run(run_name=modus):
        hits = 0
        rr_sum = 0.0
        total_time = 0.0
        
        for i, (query, target_id) in enumerate(zip(queries, target_ids)):
            start = time.time()
            
            if modus == "Modus_1_Pure_Dense":
                results = search_pure_dense(query)
            elif modus == "Modus_2_Pure_Keyword":
                results = search_pure_keyword(query)
            else:
                results = search_hybrid(query)
            
            elapsed = time.time() - start
            total_time += elapsed
            
            retrieved_ids = []
            for r in results:
                payload = r.get("payload", {})
                if payload and "paper_id" in payload:
                    retrieved_ids.append(payload["paper_id"])
            
            if target_id in retrieved_ids:
                hits += 1
                rank = retrieved_ids.index(target_id) + 1
                rr_sum += 1.0 / rank
        
        hit_rate = hits / len(queries) * 100
        mrr = rr_sum / len(queries)
        avg_latency = total_time / len(queries)
        
        print(f"📊 Hit-Rate: {hit_rate:.2f}%")
        print(f"📊 MRR: {mrr:.4f}")
        print(f"📊 Latenz: {avg_latency:.2f}s")
        
        mlflow.log_metric("hit_rate", hit_rate)
        mlflow.log_metric("mrr", mrr)
        mlflow.log_metric("latency", avg_latency)

---
## 6. Ergebnisse

| Suchmodus | Hit Rate (%) | MRR | Latenz (s) |
|-----------|-------------|-----|------------|
| **Pure Dense** | 70,71 | 0,593 | 0,81 |
| **Pure Keyword** | 0,00 | 0,000 | 0,62 |
| **Dense + Keyword Hybrid** | 70,71 | 0,593 | 0,81 |

### Interpretation
1. **Pure Keyword versagt vollständig** – 0% Hit Rate
2. **Dense und Hybrid sind identisch** – 70,71% Hit Rate
3. **Hybrid hat keinen Latenz-Overhead** – < 0,02s Unterschied

---
## 7. Hybrid Search Live Example

Ein Live-Test mit der Frage:
> "How do multi-step web agents interact with external tools and simulation servers?"

Filter: `"MCP"`

Ergebnis: Server-Latenz von **0,3954 Sekunden** über 425.310 Chunks.

In [ ]:
import time
import requests

# ==========================================
# 1. USER QUERY DEFINIEREN & ENKODIEREN
# ==========================================
user_query = "How do multi-step web agents interact with external tools and simulation servers?"

print(f"🔮 Enkodiere Abfrage via SentenceTransformer: '{user_query}'")

# Generiere den dichten Vektor (SentenceTransformer gibt ein 2D-Array zurück)
raw_vector = model.encode([user_query])

# 🛠️ DIE ENTSCHEIDENDE REPARATUR: Nimm das erste Element, um eine flache 1D-Liste (1024-d) zu erhalten!
if hasattr(raw_vector, "tolist"):
    dense_vector = raw_vector.tolist()[0]
else:
    dense_vector = list(raw_vector)[0]

# Sicherheits-Check für deine Dokumentation (Muss exakt 1024 ausgegeben werden)
print(f"📐 Vektor-Dimension erfolgreich harmonisiert: {len(dense_vector)} Elemente.")

# ==========================================
# 2. DIREKTER HTTP-POST-REQUEST GEGEN QDRANT
# ==========================================
print("📡 Feure dichte Vektorsuche via nativem HTTP-REST-Protokoll...")

base_url = url.rstrip('/')
search_endpoint = f"{base_url}/collections/stage2_docling_hybrid_bge/points/search"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# Das exakte, flache JSON-Payload, das dein Server erwartet
json_payload = {
    "vector": dense_vector,  # Jetzt eine saubere, flache 1D-Liste
    "filter": {
        "must": [
            {
                "key": "text_llm",
                "match": {
                    "text": "MCP"  # Volltext-Keyword-Filter für 'MCP'
                }
            }
        ]
    },
    "limit": 5,
    "with_payload": True
}

search_result = []

try:
    start_time = time.time()
    response = requests.post(search_endpoint, json=json_payload, headers=headers, timeout=30)
    latency = time.time() - start_time
    
    if response.status_code == 200:
        response_data = response.json()
        search_result = response_data.get("result", [])
    else:
        print(f"❌ Server antwortete mit Fehler {response.status_code}: {response.text}")
except Exception as http_err:
    print(f"❌ Fehler bei der Netzwerk-Übertragung: {http_err}")

# ==========================================
# 3. ERGEBNISSE VISUALISIEREN
# ==========================================
if search_result:
    print("\n" + "="*70 + f"\n🚀 HYBRID-RESULTATE VIA HTTP-REST FÜR: '{user_query}'\n" + "="*70)
    print(f"⏱️ Server-Antwortzeit: {latency:.4f} Sekunden")
    print("-" * 70)

    for idx, hit in enumerate(search_result):
        payload = hit.get("payload", {})
        score = hit.get("score", 0.0)
        
        print(f"🔥 [RANG {idx+1}] | 📐 Semantischer Kosinus-Score: {score:.4f}")
        print(f"📄 Paper-ID:  {payload.get('paper_id')}")
        print(f"📋 Abschnitt: {payload.get('section', 'N/A')}")
        print("-" * 70)
        print(f"📝 Extrahierter Chunk:\n{payload.get('text_llm', '')[:350]}...")
        print("=" * 70)
else:
    print("⚠️ Keine Suchergebnisse zurückgegeben. Bitte überprüfe das Collection-Schema.")


---
## 8. Zusammenfassung

### Empfohlene Konfiguration
```python
BEST_SEARCH_CONFIG = {
    "mode": "Hybrid (Dense + Keyword)",
    "collection": "stage2_docling_hybrid_bge",
    "top_k": 10,
    "embedding": "BAAI/bge-m3",
    "hit_rate": 70.71,
    "mrr": 0.593,
    "latency": 0.81,
}
```

### Wissenschaftliche Erkenntnisse
1. **Keyword-Suche versagt** bei wissenschaftlichen Fragen (0% Hit Rate)
2. **Dense + Hybrid sind identisch** – semantische Suche ist ausreichend
3. **Hybrid ist sicherer** – kein Latenz-Overhead, gleiche Qualität